<a href="https://colab.research.google.com/github/Shresthanirjala/Complete-Python-Bootcamp/blob/main/FaQ_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Run this cell once, then go to Runtime > Restart session
!pip install -q sentence-transformers scikit-learn openpyxl pandas numpy spacy ipywidgets
!python -m spacy download en_core_web_lg -q
print("Installation complete. Go to Runtime > Restart session, then run the remaining cells.")

In [ ]:
import re, io, time
import numpy as np
import pandas as pd
import spacy
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from google.colab import files

print("All imports successful.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — IMPORTS & GLOBAL CONSTANTS                            ║
# ╚══════════════════════════════════════════════════════════════════╝


#  SentenceTransformer, AgglomerativeClustering, widgets, etc.)

ENTITY_PLACEHOLDERS = {
    'PERSON':      'PERSON_NAME',
    'ORG':         'COMPANY_NAME',
    'PRODUCT':     'PRODUCT_NAME',
    'GPE':         'LOCATION_NAME',
    'LOC':         'LOCATION_NAME',
    'FAC':         'LOCATION_NAME',
    'MONEY':       'AMOUNT',
    'CARDINAL':    'NUMBER',
    'ORDINAL':     'NUMBER',
    'DATE':        'DATE_VALUE',
    'TIME':        'TIME_VALUE',
    'PERCENT':     'PERCENTAGE',
    'LAW':         'POLICY_NAME',
    'EVENT':       'EVENT_NAME',
    'WORK_OF_ART': 'CONTENT_NAME',
    'NORP':        'GROUP_NAME',
    'LANGUAGE':    'LANGUAGE_NAME',
}

STRUCTURAL_FILTER_WORDS = {
    'a', 'an', 'the', 'this', 'that', 'these', 'those', 'your', 'our', 'their', 'its',
    'please', 'kindly', 'only', 'then', 'just', 'also', 'here', 'there', 'now',
    'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'from', 'into',
    'is', 'are', 'was', 'were', 'be'
}

_nlp   = None
_model = None

In [ ]:
def get_nlp():
    global _nlp

    if _nlp is None:
        print('  Loading spaCy en_core_web_lg...', end=' ', flush=True)

        # Load from disk, or download automatically if not installed
        try:
            _nlp = spacy.load('en_core_web_lg')
        except OSError:
            import subprocess, sys
            subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_lg', '-q'])
            _nlp = spacy.load('en_core_web_lg')

        # Enable only needed pipeline components to save memory and speed up processing
        _nlp.select_pipes(enable=['tok2vec', 'tagger', 'parser', 'ner'])
        print('done')

    return _nlp


def get_embed_model():
    global _model

    if _model is None:
        print('  Loading sentence-transformer...', end=' ', flush=True)

        # BGE-large converts text into dense vectors for semantic search and similarity
        _model = SentenceTransformer('BAAI/bge-large-en-v1.5')
        print('done')

    return _model

In [ ]:
def clean_and_normalize(text, doc):

    # 1. Mask Named Entities
    # Sorted in reverse order so replacing characters doesn't shift positions of earlier entities
    entities = sorted(doc.ents, key=lambda e: e.start_char, reverse=True)
    text_chars = list(text)
    for ent in entities:
        ph = ENTITY_PLACEHOLDERS.get(ent.label_)
        if ph:
            # Swap entity span (e.g "John") with placeholder (e.g "PERSON")
            text_chars[ent.start_char:ent.end_char] = list(ph)
    masked_text = "".join(text_chars)

    # 2. Re-process masked text for token filtering
    nlp = get_nlp()
    clean_doc = nlp(masked_text)

    kept_tokens = []
    for token in clean_doc:
        token_lower = token.text.lower()
        if token_lower in STRUCTURAL_FILTER_WORDS:
            # Keep noise words only if they serve an important grammatical role
            if token.dep_ in ('mark', 'attr') or token.pos_ in ('NOUN', 'VERB'):
                kept_tokens.append(token.text)
            continue
        kept_tokens.append(token.text)

    cleaned_str = " ".join(kept_tokens)
    # Fix spacing artifacts before punctuation e.g. "Hello ." → "Hello."
    cleaned_str = re.sub(r'\s+([?.!,;:])', r'\1', cleaned_str)

    # Fallback to original text if cleaning produced an empty result
    return cleaned_str if cleaned_str.strip() else text

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — TOPIC TITLE GENERATOR                                 ║
# ║  Builds a human-readable cluster label from TF-IDF keywords.   ║
# ╚══════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────
#  SECTION 4 — TOPIC TITLE GENERATOR
# ─────────────────────────────────────────────────────────────

def make_topic_title(questions, top_n=5):
    if len(questions) == 1:
        return questions[0]
    try:
        vec   = TfidfVectorizer(stop_words='english', max_features=400, ngram_range=(1, 2))
        tfidf = vec.fit_transform(questions)
        scores = np.asarray(tfidf.sum(axis=0)).flatten()
        terms  = [vec.get_feature_names_out()[i] for i in scores.argsort()[::-1][:top_n]]
        return ' | '.join(terms).title()
    except Exception:
        return questions[0]

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — EXCEL EXPORT                                          ║
# ╚══════════════════════════════════════════════════════════════════╝

def export_faq_excel(final_faqs, out_path):
    import openpyxl
    from openpyxl.styles import Alignment, PatternFill, Font, Border, Side
    from openpyxl.utils import get_column_letter

    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "FAQ Clusters"

    # ── removed Rank, Num_Clients and Clients ──
    headers = [
        "FAQ_Title",
        "Representative_Q",
        "Cluster_Size",
        "Avg_Pairwise_Similarity",
        "Similar Questions  [pairwise_sim% | question]",
    ]

    header_fill = PatternFill("solid", fgColor="2D3748")
    header_font = Font(bold=True, color="FFFFFF", size=11)
    thin   = Side(style="thin", color="D1D5DB")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    for col_idx, h in enumerate(headers, 1):
        cell           = ws.cell(row=1, column=col_idx, value=h)
        cell.fill      = header_fill
        cell.font      = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border    = border

    ws.row_dimensions[1].height = 30

    fill_light = PatternFill("solid", fgColor="F9FAFB")
    fill_dark  = PatternFill("solid", fgColor="EBF4FF")

    for row_idx, faq in enumerate(final_faqs, 2):
        fill = fill_light if row_idx % 2 == 0 else fill_dark

        # ── percentage format ──
        stacked_qs = "\n".join(
            f"{i+1}. [{score*100:.1f}%] {q}"
            for i, (q, score) in enumerate(faq["Similar_Questions"])
        )

        row_data = [
            faq["FAQ_Title"],
            faq["Representative_Q"],
            faq["Cluster_Size"],
            f'{faq["Avg_Similarity"]*100:.1f}%',
            stacked_qs,
        ]

        row_height = max(40, len(faq["Similar_Questions"]) * 16)

        for col_idx, value in enumerate(row_data, 1):
            cell           = ws.cell(row=row_idx, column=col_idx, value=value)
            cell.fill      = fill
            cell.border    = border
            cell.alignment = Alignment(
                vertical="top", wrap_text=True,
                horizontal="center" if col_idx in (3, 4) else "left"
            )
            cell.font = Font(bold=True, size=10) if col_idx == 3 else Font(size=10)

        ws.row_dimensions[row_idx].height = row_height

    # ── 5 columns now ──
    col_widths = [28, 42, 13, 22, 62]
    for col_idx, width in enumerate(col_widths, 1):
        ws.column_dimensions[get_column_letter(col_idx)].width = width

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = f"A1:{get_column_letter(len(headers))}1"
    wb.save(out_path)
    print(f"\nExcel saved -> {out_path}  ({len(final_faqs)} FAQ groups)")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — UI WIDGETS                                            ║
# ╚══════════════════════════════════════════════════════════════════╝

display(HTML("""
<style>
  .section-title { font-size:15px; font-weight:bold; color:#1a73e8;
    margin:14px 0 4px 0; border-bottom:2px solid #1a73e8; padding-bottom:3px; }
  .hint { font-size:11px; color:#888; margin-top:2px; }
</style>
"""))

display(HTML('<div class="section-title">Step 1 - Upload your Excel file</div>'))
upload_widget = widgets.FileUpload(
    accept='.xlsx,.xls',
    multiple=False,
    description='Upload Excel',
    layout=widgets.Layout(width='260px'),
)
display(upload_widget)
display(HTML('<div class="hint">File must have a Client column and a Questions column</div>'))

display(HTML('<div class="section-title">Step 2 - Column names in your file</div>'))

col_client = widgets.Text(
    value='Client',
    description='Client col:',
    layout=widgets.Layout(width='300px'),
    style={'description_width': '100px'},
)

col_question = widgets.Text(
    value='Questions',
    description='Question col:',
    layout=widgets.Layout(width='300px'),
    style={'description_width': '100px'},
)

display(col_client, col_question)
display(HTML('<div class="hint">Change these to match your actual column headers</div>'))

display(HTML('<div class="section-title">Step 3 - Clustering settings</div>'))

sim_threshold = widgets.FloatSlider(
    value=0.12,
    min=0.05,
    max=0.30,
    step=0.01,
    description='Merge sensitivity:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px'),
    readout_format='.2f',
)

display(sim_threshold)
display(HTML("""
<div class="hint">
Low (0.05-0.10) = very strict, only nearly identical questions merge.<br>
Mid (0.11-0.16) = recommended, merges questions with the same meaning.<br>
High (0.17-0.30) = more aggressive, merges broadly similar questions.<br>
<b>Recommended starting value: 0.12</b>
</div>
"""))

top_n_output = widgets.IntSlider(
    value=20,
    min=5,
    max=100,
    step=5,
    description='Show top N FAQs:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px'),
)

display(top_n_output)

display(HTML('<div class="section-title">Step 4 - Output filename</div>'))

output_filename = widgets.Text(
    value='faq_output.xlsx',
    description='Save as:',
    layout=widgets.Layout(width='320px'),
    style={'description_width': '80px'},
)

display(output_filename)
display(HTML('<div style="margin-top:16px"></div>'))

run_button = widgets.Button(
    description='Run FAQ Clustering',
    button_style='primary',
    layout=widgets.Layout(width='220px', height='40px'),
)

out = widgets.Output()
display(run_button, out)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — MAIN PIPELINE + RUN-BUTTON WIRING                     ║
# ║  Defines the full 6-step pipeline and attaches it to the       ║
# ║  Run button created in Cell 6.                                  ║
# ║  Must be run AFTER Cell 6 so run_button/out already exist.     ║
# ╚══════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────
#  SECTION 7 — MAIN PIPELINE
# ─────────────────────────────────────────────────────────────

def run_pipeline(_):
    with out:
        from IPython.display import clear_output
        clear_output(wait=True)

        if not upload_widget.value:
            print('ERROR: No file uploaded. Please upload an Excel file first.')
            return

        uploaded_file = list(upload_widget.value.values())[0]
        file_bytes    = uploaded_file['content']
        client_col    = col_client.value.strip()
        question_col  = col_question.value.strip()

        sim_thresh    = sim_threshold.value
        top_n         = top_n_output.value
        out_path      = output_filename.value.strip() or 'faq_output.xlsx'

        print('=' * 65)
        print('  FAQ SEMANTIC CLUSTERING PIPELINE')
        print('=' * 65)

        # ── [1/6] Load data ──────────────────────────────────────────
        print('\n[1/6] Loading file...', end=' ')
        try:
            df = pd.read_excel(io.BytesIO(bytes(file_bytes)))
        except Exception as e:
            print(f'\nERROR: Could not read file: {e}')
            return

        missing = [c for c in [client_col, question_col] if c not in df.columns]
        if missing:
            print(f'\nERROR: Column(s) not found: {missing}')
            return

        df = df.rename(columns={client_col: '_Client', question_col: '_Question'})
        df = df.dropna(subset=['_Client', '_Question'])
        # Normalize whitespace and remove duplicate client-question pairs
        df['_Question'] = (df['_Question'].astype(str).str.strip()
                           .str.replace(r'\s+', ' ', regex=True))
        df = df.drop_duplicates(subset=['_Client', '_Question']).reset_index(drop=True)
        print(f'done  ->  {len(df):,} items')

        # ── [2/6] Load models ────────────────────────────────────────
        print('\n[2/6] Loading models...')
        nlp   = get_nlp()
        model = get_embed_model()

        # ── [3/6] Clean & normalize ──────────────────────────────────
        print('\n[3/6] Filtering conversational noise and mapping entities...', end=' ', flush=True)
        t0 = time.time()

        # Process in batches of 256 for memory efficiency
        texts, processed_texts = df['_Question'].tolist(), []
        for i in range(0, len(texts), 256):
            batch = texts[i:i+256]
            docs  = list(nlp.pipe(batch))
            for doc, orig in zip(docs, batch):
                processed_texts.append(clean_and_normalize(orig, doc))

        df['_Question_Cleaned'] = processed_texts
        print(f'done in {time.time()-t0:.1f}s')

        print('\n   Sample Transformation Context:')
        for _, row in df[['_Question', '_Question_Cleaned']].head(3).iterrows():
            print(f'   Before : {row["_Question"]}')
            print(f'   After  : {row["_Question_Cleaned"]}\n')

        # ── [4/6] Embeddings ─────────────────────────────────────────
        print('[4/6] Generating Deep Semantic Embeddings...', end=' ', flush=True)
        t0 = time.time()
        # normalize_embeddings=True makes cosine similarity a simple dot product (faster)
        all_embeddings = model.encode(
            df['_Question_Cleaned'].tolist(),
            batch_size=64, show_progress_bar=False, normalize_embeddings=True,
        )
        df['_emb_idx'] = range(len(df))
        print(f'done in {time.time()-t0:.1f}s')

        # ── [5/6] Clustering ─────────────────────────────────────────
        print(f'\n[5/6] Grouping similar questions (distance_threshold={sim_thresh})...')
        all_faqs = []

        cdf  = df.copy().reset_index(drop=True)
        embs = all_embeddings[cdf['_emb_idx'].values]

        if len(cdf) < 2:
            # Edge case: only one question in the file, skip clustering
            all_faqs.append({
                'FAQ_Title':         cdf.loc[0, '_Question'],
                'Representative_Q':  cdf.loc[0, '_Question'],
                'Cluster_Size':      1,
                'Avg_Similarity':    0.0,
                'Similar_Questions': [(cdf.loc[0, '_Question'], 0.0)],
            })
        else:
            # Agglomerative clustering groups questions by cosine distance
            # distance_threshold controls how similar questions must be to merge
            clustering = AgglomerativeClustering(
                n_clusters=None,
                metric='cosine',
                linkage='average',
                distance_threshold=sim_thresh
            )
            labels = clustering.fit_predict(embs)
            cdf['Cluster'] = labels

            for cluster_id in np.unique(labels):
                cdf_c  = cdf[cdf['Cluster'] == cluster_id]
                c_embs = embs[cdf_c.index]
                qs     = cdf_c['_Question'].tolist()

                # Pick the question closest to the cluster centroid as the representative
                centroid = c_embs.mean(axis=0, keepdims=True)
                sims     = cosine_similarity(c_embs, centroid).flatten()
                rep_idx  = sims.argmax()
                rep_q    = cdf_c.iloc[rep_idx]['_Question']

                if len(c_embs) > 1:
                    pairwise = cosine_similarity(c_embs)
                    n = len(qs)

                    # Each question's average similarity to every other question in cluster
                    per_q_avg_sims = [
                        float((pairwise[i].sum() - 1.0) / max(n - 1, 1))
                        for i in range(n)
                    ]

                    # Cluster cohesion score — mean of upper triangle avoids counting pairs twice
                    upper = pairwise[np.triu_indices(n, k=1)]
                    avg_pairwise_sim = float(upper.mean()) if len(upper) > 0 else 0.0

                else:
                    # Single-question cluster — no similarity to compute
                    per_q_avg_sims   = [0.0]
                    avg_pairwise_sim = 0.0

                q_sim_pairs = sorted(
                    zip(qs, per_q_avg_sims),
                    key=lambda x: x[1], reverse=True
                )

                all_faqs.append({
                    'FAQ_Title':         make_topic_title(qs),
                    'Representative_Q':  rep_q,
                    'Cluster_Size':      len(qs),
                    'Avg_Similarity':    round(avg_pairwise_sim, 4),
                    'Similar_Questions': list(q_sim_pairs),
                })

        # Rank clusters by cohesion first, then size — most useful clusters appear on top
        all_faqs.sort(
            key=lambda x: (x['Avg_Similarity'], x['Cluster_Size']),
            reverse=True
        )

        print(f'   -> Created {len(all_faqs):,} clusters.')

        # ── [6/6] Print & export ─────────────────────────────────────
        print(f'\n[6/6] Results Summary\n')
        print('=' * 65)
        print(f'  TOP {min(top_n, len(all_faqs))} FAQ CLUSTERS  '
              f'(ranked by avg pairwise similarity)')
        print('=' * 65)

        for rank, faq in enumerate(all_faqs[:top_n], 1):
            print(f'\n  #{rank}  Cluster Size: {faq["Cluster_Size"]} | '
                  f'Avg Pairwise Sim: {faq["Avg_Similarity"]*100:.1f}%')
            print(f'  Representative Q : {faq["Representative_Q"]}')
            print('  All Questions (avg pairwise similarity to others in cluster):')
            for q, score in faq['Similar_Questions']:
                bar = '█' * int(score * 20)
                print(f'    [{score*100:.1f}%] {bar:<20}  {q}')
            print('-' * 65)

        export_faq_excel(all_faqs[:top_n], out_path)

    # Outside the output widget so the download dialog triggers in Colab correctly
    from google.colab import files
    import os
    if os.path.exists(out_path):
        files.download(out_path)
    else:
        with out:
            print(f'\nERROR: File not found — {out_path}')

run_button.on_click(run_pipeline)

In [ ]:
# ── Wire the button to the pipeline (must be in the same cell as run_pipeline) ──
run_button.on_click(run_pipeline)
print('Pipeline configured successfully with pairwise semantic similarity ranking.')


In [ ]:
# --- UI Layout Components for Dual File Upload and Analysis ---
upload_file_1 = widgets.FileUpload(accept='.xlsx,.xls', multiple=False, description='Upload Client 1 File', layout=widgets.Layout(width='220px'))
upload_file_2 = widgets.FileUpload(accept='.xlsx,.xls', multiple=False, description='Upload Client 2 File', layout=widgets.Layout(width='220px'))
col_target = widgets.Text(value='Questions', description='Question Column:', layout=widgets.Layout(width='280px'), style={'description_width': 'initial'})

local_sensitivity = widgets.FloatSlider(value=0.14, min=0.05, max=0.30, step=0.01, description='Local Sensitivity:', layout=widgets.Layout(width='450px'), style={'description_width': 'initial'})
cross_strictness = widgets.FloatSlider(value=0.90, min=0.75, max=0.99, step=0.01, description='Cross-Match Cutoff:', layout=widgets.Layout(width='450px'), style={'description_width': 'initial'})

dual_execute_btn = widgets.Button(description='Process & Extract Common FAQs', button_style='success', layout=widgets.Layout(width='280px', height='40px'))
dual_display_out = widgets.Output()

def extract_client_faqs(dataframe, question_column, sensitivity_threshold):
    """
    Cleans raw text, generates BGE embeddings, clusters them locally,
    and returns independent, aggregated FAQ patterns for a given dataframe.
    """
    # Standardize data structure
    df_clean = dataframe[[question_column]].dropna().drop_duplicates().reset_index(drop=True)
    raw_texts = df_clean[question_column].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True).tolist()

    # Dynamic runtime fallback for Cell 1 function names
    globals_dict = globals()
    if 'get_nlp' in globals_dict:
        nlp_instance = globals_dict['get_nlp']()
    elif 'get_shared_nlp' in globals_dict:
        nlp_instance = globals_dict['get_shared_nlp']()
    else:
        import spacy
        nlp_instance = spacy.load('en_core_web_lg')

    if 'get_embed_model' in globals_dict:
        model_instance = globals_dict['get_embed_model']()
    elif 'get_shared_model' in globals_dict:
        model_instance = globals_dict['get_shared_model']()
    else:
        from sentence_transformers import SentenceTransformer
        model_instance = SentenceTransformer('BAAI/bge-large-en-v1.5')

    # Process text through the structural cleanup layer from Cell 1
    processed_texts = []
    for i in range(0, len(raw_texts), 256):
        batch = raw_texts[i:i+256]
        docs = list(nlp_instance.pipe(batch))
        for doc, orig in zip(docs, batch):
            processed_texts.append(clean_and_normalize(orig, doc))

    df_clean['_Cleaned'] = processed_texts

    # Generate vector weights using BGE-Large
    embeddings = model_instance.encode(df_clean['_Cleaned'].tolist(), batch_size=64, show_progress_bar=False, normalize_embeddings=True)

    if len(df_clean) < 2:
        return [{'rep_q': df_clean.loc[0, question_column], 'freq': 1, 'vector': embeddings[0]}]

    # Cluster the text to identify internal frequently asked questions
    clustering = AgglomerativeClustering(n_clusters=None, metric='cosine', linkage='average', distance_threshold=sensitivity_threshold)
    labels = clustering.fit_predict(embeddings)
    df_clean['Cluster'] = labels

    faq_patterns = []
    for c_id in np.unique(labels):
        cluster_rows = df_clean[df_clean['Cluster'] == c_id]
        cluster_embeddings = embeddings[cluster_rows.index]

        # Calculate cluster centroid to pick the best representative text
        centroid = cluster_embeddings.mean(axis=0, keepdims=True)
        similarities = cosine_similarity(cluster_embeddings, centroid).flatten()
        representative_text = cluster_rows.iloc[similarities.argmax()][question_column]

        faq_patterns.append({
            'rep_q': representative_text,
            'freq': len(cluster_rows),
            'vector': centroid.flatten()
        })

    return faq_patterns

def process_dual_file_matching(_):
    with dual_display_out:
        from IPython.display import clear_output
        clear_output(wait=True)

        if not upload_file_1.value or not upload_file_2.value:
            print("ERROR: Please upload separate spreadsheets in both file fields above.")
            return

        target_column = col_target.value.strip()
        local_thresh = local_sensitivity.value
        match_thresh = cross_strictness.value

        print("=" * 75)
        print(" DUAL-FILE INDEPENDENT CLUSTERING & CROSS-MATCH ENGINE")
        print("=" * 75)

        print("\n[1/4] Parsing raw uploaded spreadsheet inputs...", end=' ', flush=True)
        try:
            f1_bytes = list(upload_file_1.value.values())[0]['content']
            f2_bytes = list(upload_file_2.value.values())[0]['content']
            df1 = pd.read_excel(io.BytesIO(bytes(f1_bytes)))
            df2 = pd.read_excel(io.BytesIO(bytes(f2_bytes)))
        except Exception as e:
            print(f"\nERROR: Excel reading operation failed: {e}")
            return

        if target_column not in df1.columns or target_column not in df2.columns:
            print(f"\nERROR: Target column '{target_column}' is missing from one or both files.")
            return
        print("Done.")

        print("[2/4] Generating independent local FAQ patterns for File 1...", end=' ', flush=True)
        raw_faqs_1 = extract_client_faqs(df1, target_column, local_thresh)
        # CRITICAL FILTER: Keep only groups where the frequency is greater than 1
        faqs_file_1 = [f for f in raw_faqs_1 if f['freq'] > 1]
        print(f"Done. (Found {len(raw_faqs_1)} total groups -> Filtered to {len(faqs_file_1)} High-Frequency FAQs)")

        print("[3/4] Generating independent local FAQ patterns for File 2...", end=' ', flush=True)
        raw_faqs_2 = extract_client_faqs(df2, target_column, local_thresh)
        # CRITICAL FILTER: Keep only groups where the frequency is greater than 1
        faqs_file_2 = [f for f in raw_faqs_2 if f['freq'] > 1]
        print(f"Done. (Found {len(raw_faqs_2)} total groups -> Filtered to {len(faqs_file_2)} High-Frequency FAQs)")

        if not faqs_file_1 or not faqs_file_2:
            print("\nExecution Halting: One or both files contain zero repeating FAQ patterns (Frequency > 1).")
            print("Try loosening your 'Local Sensitivity' slider to encourage broader internal clustering.")
            return

        print("[4/4] Evaluating cross-file semantic alignment patterns...", end=' ', flush=True)
        vectors_1 = np.array([f['vector'] for f in faqs_file_1])
        vectors_2 = np.array([f['vector'] for f in faqs_file_2])

        sim_matrix = cosine_similarity(vectors_1, vectors_2)
        print("Done.")

        matched_results = []
        visited_file_2 = set()

        for idx1, row_scores in enumerate(sim_matrix):
            best_idx2 = row_scores.argmax()
            best_score = row_scores[best_idx2]

            if best_score >= match_thresh and best_idx2 not in visited_file_2:
                matched_results.append({
                    'alignment': best_score,
                    'faq_1': faqs_file_1[idx1]['rep_q'],
                    'freq_1': faqs_file_1[idx1]['freq'],
                    'faq_2': faqs_file_2[best_idx2]['rep_q'],
                    'freq_2': faqs_file_2[best_idx2]['freq']
                })
                visited_file_2.add(best_idx2)

        if not matched_results:
            print(f"\nNo shared recurrent FAQ themes found between the two files at a {match_thresh*100:g}% similarity threshold.")
            print("Tip: Adjust your sliders to fine-tune the strictness settings.")
            return

        matched_results.sort(key=lambda x: x['alignment'], reverse=True)

        print("\n" + "=" * 75)
        print(f" SUCCESS: EXTRACTED {len(matched_results)} MUTUAL HIGH-FREQUENCY FAQs")
        print("=" * 75)

        for rank, match in enumerate(matched_results, 1):
            print(f"\n[Shared FAQ Theme #{rank}]  (Semantic Match: {match['alignment']*100:.1f}%)")
            print(f"  ▪ File 1 (Internal Repeat Count: {match['freq_1']}) ➔ \"{match['faq_1']}\"")
            print(f"  ▪ File 2 (Internal Repeat Count: {match['freq_2']}) ➔ \"{match['faq_2']}\"")
            print("-" * 75)

# Bind functionality to button execution trigger
dual_execute_btn.on_click(process_dual_file_matching)

# Layout Assembly and Display
display(widgets.VBox([
    widgets.HTML("<br><hr><h3>Dual-Client Separate File Overlap Engine (High-Frequency Filtered)</h3>"),
    widgets.HTML("<h4>Step 1: Upload independent client files and specify target column:</h4>"),
    widgets.HBox([upload_file_1, upload_file_2, col_target]),
    widgets.HTML("<h4>Step 2: Define strictness rules for local grouping and cross-matching:</h4>"),
    widgets.VBox([local_sensitivity, cross_strictness]),
    widgets.Box([dual_execute_btn], style={'margin': '15px 0px'}),
    dual_display_out
]))

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the model
model = SentenceTransformer("BAAI/bge-large-en-v1.5")

# Define your text
# sentences = ["Were any records, systems, or data assets subject to a cybersecurity incident within the previous five-year period?", "During the past years, have there been any security incidents resulting in the disclosure, alteration, or destruction of sensitive data? Please explain if applicable."]
sentences = ["Were any records, systems, or data assets subject to a cybersecurity incident within the previous five-year period?", "Has your organization suffered a physical break-in during the past five years? If so, please provide details."]

# Generate the embeddings
embeddings = model.encode(sentences)

# Calculate how similar they are
similarities = model.similarity(embeddings, embeddings)
print(similarities)


In [ ]:
from sentence_transformers import SentenceTransformer

# Load the model
model = SentenceTransformer("BAAI/bge-large-en-v1.5")

# Define your text
# sentences = ["Were any records, systems, or data assets subject to a cybersecurity incident within the previous five-year period?", "During the past years, have there been any security incidents resulting in the disclosure, alteration, or destruction of sensitive data? Please explain if applicable."]
sentences = ["Were any records, systems, or data assets subject to a cybersecurity incident within the previous five-year period?", "Has your organization suffered a physical break-in during the past five years? If so, please provide details."]

# Generate the embeddings
embeddings = model.encode(sentences)

# Calculate how similar they are
similarities = model.similarity(embeddings, embeddings)
print(similarities)


In [ ]:
from sentence_transformers import SentenceTransformer

# Load the model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Define your text
sentences = ["Were any records, systems, or data assets subject to a cybersecurity incident within the previous five-year period?", "During the past years, have there been any security incidents resulting in the disclosure, alteration, or destruction of sensitive data? Please explain if applicable."]
# sentences = ["Were any records, systems, or data assets subject to a cybersecurity incident within the previous five-year period?", "Has your organization suffered a physical break-in during the past five years? If so, please provide details."]

# Generate the embeddings
embeddings = model.encode(sentences)

# Calculate how similar they are
similarities = model.similarity(embeddings, embeddings)
print(similarities)
